# Q1. Ingestion Snowflake

**Objectif :** Charger les fichiers WAV dans un Stage Snowflake et stocker les métadonnées dans une table structurée.

**Architecture :**
```
┌─────────────────────────────────────────────────────┐
│  Snowflake                                          │
│                                                     │
│  Stage @RESPIRATORY_STAGE                           │
│    ├── bronchial/fichier.wav                        │
│    ├── asthma/fichier.wav  ...                      │
│    └── model.pth  (uploadé après entraînement)      │
│                                                     │
│  Table AUDIO_METADATA                               │
│    filename | label | duration_s | sample_rate | .. │
│                                                     │
│  Table PREDICTIONS (inference ultérieure)           │
│    timestamp | label_pred | probas | confiance      │
└─────────────────────────────────────────────────────┘
```

In [1]:
import os
import librosa
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from snowflake.snowpark import Session
from snowflake.snowpark.types import (
    StructType, StructField,
    StringType, FloatType, IntegerType, LongType, TimestampType
)

load_dotenv('../.env', override=True)
print('Librairies chargées.')

Librairies chargées.


In [2]:
DATA_DIR      = '../data'
MIN_DURATION  = 1.0       # secondes — drop fichiers trop courts
STAGE_NAME    = 'RESPIRATORY_STAGE'
TABLE_META    = 'AUDIO_METADATA'
TABLE_PRED    = 'PREDICTIONS'

## Connexion Snowflake

In [4]:
import getpass

totp = getpass.getpass("Code MFA (6 chiffres) : ")

connection_params = {
    'account'      : os.environ['SNOWFLAKE_ACCOUNT'],
    'user'         : os.environ['SNOWFLAKE_USER'],
    'password'     : os.environ['SNOWFLAKE_PASSWORD'],
    'authenticator': os.environ['SNOWFLAKE_AUTHENTICATOR'],
    'passcode'     : totp,
    'warehouse'    : os.environ['SNOWFLAKE_WAREHOUSE'],
    'database'     : os.environ['SNOWFLAKE_DATABASE'],
    'schema'       : os.environ['SNOWFLAKE_SCHEMA'],
    'role'         : os.environ['SNOWFLAKE_ROLE'],
}

session = Session.builder.configs(connection_params).create()
print(f"Connecté : {session.get_current_database()}.{session.get_current_schema()}")
print(f"Warehouse : {session.get_current_warehouse()}")

 pip install snowflake-connector-python[secure-local-storage]
 pip install snowflake-connector-python[secure-local-storage]


DatabaseError: 250001 (08001): Failed to connect to DB: SFEDU02-DYB15780.snowflakecomputing.com:443. TOTP Invalid.

## Création du Stage et des Tables

- **Stage interne** : stocke les fichiers WAV bruts
- **AUDIO_METADATA** : métadonnées extraites localement avant upload
- **PREDICTIONS** : recevra les résultats d'inférence du modèle

In [4]:
session.sql(f"CREATE STAGE IF NOT EXISTS {STAGE_NAME} COMMENT = 'Fichiers WAV bruts — Asthma Detection Dataset v2'").collect()

session.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_META} (
        FILENAME      VARCHAR(255) NOT NULL,
        LABEL         VARCHAR(50)  NOT NULL,
        DURATION_S    FLOAT        NOT NULL,
        SAMPLE_RATE   INTEGER      NOT NULL,
        N_SAMPLES     INTEGER      NOT NULL,
        FILE_SIZE_KB  FLOAT,
        STAGE_PATH    VARCHAR(500)
    )
""").collect()

session.sql(f"""
    CREATE OR REPLACE TABLE {TABLE_PRED} (
        PREDICTION_ID  VARCHAR(36)   DEFAULT UUID_STRING(),
        TIMESTAMP      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
        PHARMACIE_ID   VARCHAR(100)  NOT NULL,
        CLASSE_PREDITE VARCHAR(50)   NOT NULL,
        PROBABILITES   VARIANT,
        CONFIANCE      FLOAT         NOT NULL
    )
""").collect()

print(f"Stage  '{STAGE_NAME}' : OK")
print(f"Table  '{TABLE_META}' : OK")
print(f"Table  '{TABLE_PRED}' : OK")

Stage  'RESPIRATORY_STAGE' : OK
Table  'AUDIO_METADATA' : OK
Table  'PREDICTIONS' : OK


## Extraction des métadonnées locales

In [5]:
records = []
skipped = 0

for label in sorted(os.listdir(DATA_DIR)):
    label_dir = os.path.join(DATA_DIR, label)
    if not os.path.isdir(label_dir):
        continue
    for fname in sorted(os.listdir(label_dir)):
        if not fname.lower().endswith('.wav'):
            continue
        fpath = os.path.join(label_dir, fname)
        duration = librosa.get_duration(path=fpath)
        if duration < MIN_DURATION:
            print(f"  Skipped (trop court) : {fname}")
            skipped += 1
            continue
        sr_native = librosa.get_samplerate(fpath)
        n_samples  = int(duration * sr_native)
        size_kb    = os.path.getsize(fpath) / 1024
        stage_path = f'@{STAGE_NAME}/{label.lower()}/{fname}'
        records.append({
            'FILENAME'    : fname,
            'LABEL'       : label.lower(),
            'DURATION_S'  : round(duration, 4),
            'SAMPLE_RATE' : sr_native,
            'N_SAMPLES'   : n_samples,
            'FILE_SIZE_KB': round(size_kb, 2),
            'STAGE_PATH'  : stage_path,
        })

df_meta = pd.DataFrame(records)
print(f"Fichiers indexés : {len(df_meta)} | Skippés : {skipped}")
df_meta.groupby('LABEL')[['DURATION_S', 'SAMPLE_RATE']].describe().round(2)

c:\Users\victor\Documents\ECOLE\5A\T2\AP1\HACKATHON FINAL\respiratory-disease-detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Skipped (trop court) : P20WheezingIE_97.wav
  Skipped (trop court) : P22WheezingRL_107.wav
  Skipped (trop court) : P2AsthmaIU_6.wav
Fichiers indexés : 1208 | Skippés : 3


DURATION_S                                       SAMPLE_RATE  \
               count  mean   std   min  25%  50%  75%  max       count   
LABEL                                                                    
asthma         285.0  5.92  0.38  2.38  6.0  6.0  6.0  6.0       285.0   
bronchial      104.0  5.86  0.68  2.00  6.0  6.0  6.0  6.0       104.0   
copd           401.0  5.99  0.09  4.94  6.0  6.0  6.0  6.0       401.0   
healthy        133.0  5.92  0.25  4.62  6.0  6.0  6.0  6.0       133.0   
pneumonia      285.0  5.87  0.38  3.07  6.0  6.0  6.0  6.0       285.0   

                                                                            
               mean       std      min      25%      50%      75%      max  
LABEL                                                                       
asthma      7658.25  11566.43   4000.0   4000.0   4000.0   4000.0  44100.0  
bronchial  38316.35  14156.30   4000.0  44100.0  44100.0  44100.0  44100.0  
copd       25300.00  20036.00   4000.0   4000.0  44100.0  44100.0  44100.0  
healthy    44100.00      0.00  44100.0  44100.0  44100.0  44100.0  44100.0  
pneumonia  38612.63  13805.83   4000.0  44100.0  44100.0  44100.0  44100.0

## Upload WAV → Stage Snowflake

Utilise la commande `PUT` de Snowpark pour envoyer les fichiers WAV dans le Stage.

In [6]:
from tqdm.auto import tqdm

errors = []

for label in sorted(os.listdir(DATA_DIR)):
    label_dir = os.path.join(DATA_DIR, label)
    if not os.path.isdir(label_dir):
        continue

    wav_files = [f for f in os.listdir(label_dir) if f.lower().endswith('.wav')]
    print(f"\n[{label.lower()}] {len(wav_files)} fichiers...")

    for fname in tqdm(wav_files, desc=label):
        fpath    = Path(label_dir) / fname
        duration = librosa.get_duration(path=str(fpath))
        if duration < MIN_DURATION:
            continue
        try:
            put_result = session.file.put(
                local_file_name = str(fpath.resolve()),
                stage_location  = f'@{STAGE_NAME}/{label.lower()}/',
                auto_compress   = False,
                overwrite       = False,
            )
        except Exception as e:
            errors.append((fname, str(e)))

print(f"\nUpload terminé | Erreurs : {len(errors)}")
if errors:
    for f, e in errors:
        print(f"  {f} → {e}")


[bronchial] 104 fichiers...


Bronchial: 100%|██████████| 104/104 [01:48<00:00,  1.04s/it]



[asthma] 288 fichiers...


asthma: 100%|██████████| 288/288 [02:51<00:00,  1.68it/s]



[copd] 401 fichiers...


...


[pneumonia] 285 fichiers...


pneumonia: 100%|██████████| 285/285 [03:15<00:00,  1.46it/s]



[processed] 0 fichiers...


processed: 0it [00:00, ?it/s]


Upload terminé | Erreurs : 0


## Insertion des métadonnées dans Snowflake

In [6]:
schema = StructType([
    StructField('FILENAME',     StringType()),
    StructField('LABEL',        StringType()),
    StructField('DURATION_S',   FloatType()),
    StructField('SAMPLE_RATE',  IntegerType()),
    StructField('N_SAMPLES',    IntegerType()),
    StructField('FILE_SIZE_KB', FloatType()),
    StructField('STAGE_PATH',   StringType()),
])

sp_df = session.create_dataframe(df_meta[[
    'FILENAME', 'LABEL', 'DURATION_S',
    'SAMPLE_RATE', 'N_SAMPLES', 'FILE_SIZE_KB', 'STAGE_PATH'
]].values.tolist(), schema=schema)

sp_df.write.mode('overwrite').save_as_table(TABLE_META)
print(f"Inséré {len(df_meta)} lignes dans {TABLE_META}")

Inséré 1208 lignes dans AUDIO_METADATA


## Vérification

In [7]:
# Résumé par classe
result = session.sql(f"""
    SELECT
        LABEL,
        COUNT(*)              AS N_FILES,
        ROUND(AVG(DURATION_S), 2) AS AVG_DURATION_S,
        MIN(SAMPLE_RATE)      AS SR_MIN,
        MAX(SAMPLE_RATE)      AS SR_MAX,
        ROUND(SUM(FILE_SIZE_KB)/1024, 1) AS TOTAL_MB
    FROM {TABLE_META}
    GROUP BY LABEL
    ORDER BY N_FILES DESC
""").to_pandas()

print(f"\n=== {TABLE_META} ===")
print(result.to_string(index=False))
print(f"\nTotal : {result['N_FILES'].sum()} fichiers, {result['TOTAL_MB'].sum():.1f} MB")


=== AUDIO_METADATA ===
    LABEL  N_FILES  AVG_DURATION_S  SR_MIN  SR_MAX  TOTAL_MB
     copd      401            5.99    4000   44100     116.2
pneumonia      285            5.87    4000   44100     123.2
   asthma      285            5.92    4000   44100      24.7
  healthy      133            5.92   44100   44100      66.2
bronchial      104            5.86    4000   44100      44.4

Total : 1208 fichiers, 374.7 MB


In [8]:
# Vérif Stage
stage_files = session.sql(f"LIST @{STAGE_NAME}").to_pandas()
print(f"Fichiers dans le Stage : {len(stage_files)}")
stage_files.head(10)

Fichiers dans le Stage : 1208


,"""name""","""size""","""md5""","""last_modified"""
0,respiratory_stage/asthma/P10AsthmaIE_49.wav,25344,ec4492fcba569e46bb629fc1e1b934f2,"Thu, 19 Mar 2026 16:42:36 GMT"
1,respiratory_stage/asthma/P10AsthmaIU_46.wav,39808,71d2e85a7ef0964ab96c2c2d78367f5a,"Thu, 19 Mar 2026 16:42:37 GMT"
2,respiratory_stage/asthma/P10AsthmaIU_50.wav,48256,d384b7a90832fd96d2a18aeb4aa23d99,"Thu, 19 Mar 2026 16:42:37 GMT"
3,respiratory_stage/asthma/P10AsthmaRL_47.wav,48256,0640f3bc393dbb1b9db293ab2a179f69,"Thu, 19 Mar 2026 16:42:38 GMT"
4,respiratory_stage/asthma/P10AsthmaRS_48.wav,48256,322229b42131ac8a4cc341728a6849c8,"Thu, 19 Mar 2026 16:42:39 GMT"
5,respiratory_stage/asthma/P11WheezingIE_53.wav,48256,324f85c27948d2a311a3bcad2073c0ce,"Thu, 19 Mar 2026 16:42:39 GMT"
6,respiratory_stage/asthma/P11WheezingIU_54.wav,48256,891297e19087329327c9eaf410d33e2e,"Thu, 19 Mar 2026 16:42:40 GMT"
7,respiratory_stage/asthma/P11WheezingRL_51.wav,47872,dd509189bcad414cd6b6cf2f482e6059,"Thu, 19 Mar 2026 16:42:40 GMT"
8,respiratory_stage/asthma/P11WheezingRL_55.wav,44544,5d5504d8b5a006cd792316eda60e1d30,"Thu, 19 Mar 2026 16:42:41 GMT"
9,respiratory_stage/asthma/P11WheezingRS_52.wav,48256,60918e78e3d4a22ba82b983f7cadda35,"Thu, 19 Mar 2026 16:42:41 GMT"


In [9]:
session.close()
print("Session Snowflake fermée.")

Session Snowflake fermée.


## Upload du modèle entraîné → Stage Snowflake

In [ ]:
import getpass
from pathlib import Path
from tqdm.auto import tqdm

totp = getpass.getpass("Code MFA (6 chiffres) : ")

connection_params = {
    'account'      : os.environ['SNOWFLAKE_ACCOUNT'],
    'user'         : os.environ['SNOWFLAKE_USER'],
    'password'     : os.environ['SNOWFLAKE_PASSWORD'],
    'authenticator': os.environ['SNOWFLAKE_AUTHENTICATOR'],
    'passcode'     : totp,
    'warehouse'    : os.environ['SNOWFLAKE_WAREHOUSE'],
    'database'     : os.environ['SNOWFLAKE_DATABASE'],
    'schema'       : os.environ['SNOWFLAKE_SCHEMA'],
    'role'         : os.environ['SNOWFLAKE_ROLE'],
}

session = Session.builder.configs(connection_params).create()
print(f"Connecté : {session.get_current_database()}.{session.get_current_schema()}")

model_path = Path('../models/best_model_convnext.pth').resolve()
size_mb = model_path.stat().st_size / 1024 / 1024
print(f"Upload {model_path.name} ({size_mb:.1f} MB)...")

session.file.put(
    local_file_name = str(model_path),
    stage_location  = f'@{STAGE_NAME}/models/',
    auto_compress   = False,
    overwrite       = True,
)
print(f"✓ {model_path.name} uploadé dans @{STAGE_NAME}/models/")

session.close()
print("Session fermée.")

c:\Users\victor\Documents\ECOLE\5A\T2\AP1\HACKATHON FINAL\respiratory-disease-detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 pip install snowflake-connector-python[secure-local-storage]


Connecté : "M2_UQAC_EQUIPE_1_DB"."PUBLIC"
Upload best_model_convnext.pth (107.0 MB)...
